# 📊 Unemployment Analysis in India — Python Project

> **Dataset**: [Kaggle — gokulrajkmv/unemployment-in-india](https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india)  
> **Period**: January 2019 – November 2020  
> **Focus**: Covid-19 Impact · State Trends · Seasonal Patterns · Policy Insights

---
### 📌 Objectives
1. Analyse unemployment rate data across all Indian states
2. Investigate the devastating impact of Covid-19 lockdown on employment
3. Identify seasonal patterns and regional disparities
4. Compare Urban vs Rural unemployment dynamics
5. Present data-driven insights for economic policy


## 1. 📦 Imports & Setup

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
%matplotlib inline

PRE='#2196F3'; COVID='#F44336'; POST='#4CAF50'; GOLD='#FF9800'; BG='#F8F9FF'
print('✅ All imports successful')

## 2. 📋 Load & Explore Dataset
> Download from Kaggle and place as `data/Unemployment_Rate_upto_11_2020.csv`, or run `python data/generate_dataset.py`

In [ ]:
df = pd.read_csv('data/Unemployment_Rate_upto_11_2020.csv')
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year
df['YearMonth'] = df['Date'].dt.to_period('M')
df['Covid_Period'] = df['Date'].apply(
    lambda d: 'Lockdown' if (d.year==2020 and d.month in [4,5])
              else ('Recovery' if (d.year==2020 and d.month>=6) else 'Pre-Covid'))

col_rate = 'Estimated Unemployment Rate (%)'
col_lpr  = 'Estimated Labour Participation Rate (%)'
col_emp  = 'Estimated Employed'

print(f'Shape: {df.shape} | States: {df["Region"].nunique()} | Missing: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Covid Period Distribution:')
print(df['Covid_Period'].value_counts())
print('\nPeriod-wise Mean Unemployment:')
print(df.groupby('Covid_Period')[col_rate].mean().round(2))

## 3. 📈 National Unemployment Trend

In [ ]:
national = df.groupby('Date')[col_rate].mean().reset_index()

fig, ax = plt.subplots(figsize=(14,5))
ax.axvspan(pd.Timestamp('2020-04-01'), pd.Timestamp('2020-05-31'), color=COVID, alpha=0.15, label='Lockdown')
ax.axvspan(pd.Timestamp('2020-06-01'), national['Date'].max(), color=POST, alpha=0.10, label='Recovery')
ax.plot(national['Date'], national[col_rate], color=PRE, lw=2.5, marker='o', markersize=5)
ax.fill_between(national['Date'], national[col_rate], alpha=0.2, color=PRE)
peak = national.loc[national[col_rate].idxmax()]
ax.annotate(f'PEAK {peak[col_rate]:.1f}%', xy=(peak['Date'], peak[col_rate]),
            xytext=(peak['Date'], peak[col_rate]+3),
            arrowprops=dict(arrowstyle='->', color=COVID), color=COVID, fontweight='bold', ha='center')
ax.set_title('India National Unemployment Rate — Jan 2019 to Nov 2020', fontsize=13, fontweight='bold')
ax.set_ylabel('Unemployment Rate (%)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. 🦠 Covid-19 Impact Analysis

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Covid-19 Impact on Unemployment', fontsize=13, fontweight='bold')
order = ['Pre-Covid','Lockdown','Recovery']; colors_p=[PRE,COVID,POST]
period_stats = df.groupby('Covid_Period')[col_rate].agg(['mean','std','median'])

bars = axes[0].bar(order, [period_stats.loc[p,'mean'] for p in order], color=colors_p, edgecolor='white', width=0.5)
for bar,v in zip(bars, [period_stats.loc[p,'mean'] for p in order]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Mean Rate by Period'); axes[0].set_ylabel('%'); axes[0].grid(axis='y',alpha=0.3)

data_vio = [df[df['Covid_Period']==p][col_rate].values for p in order]
vp = axes[1].violinplot(data_vio, positions=[1,2,3], showmedians=True)
for body,c in zip(vp['bodies'],colors_p): body.set_facecolor(c); body.set_alpha(0.7)
axes[1].set_xticks([1,2,3]); axes[1].set_xticklabels(order)
axes[1].set_title('Rate Distribution'); axes[1].grid(axis='y',alpha=0.3)

data_lpr = [df[df['Covid_Period']==p][col_lpr].values for p in order]
bp = axes[2].boxplot(data_lpr, labels=order, patch_artist=True, medianprops=dict(color='black',lw=2))
for patch,c in zip(bp['boxes'],colors_p): patch.set_facecolor(c); patch.set_alpha(0.7)
axes[2].set_title('Labour Participation Rate'); axes[2].grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.show()

## 5. 🗺️ State-Wise Heatmap

In [ ]:
state_monthly = df.pivot_table(index='Region', columns='YearMonth', values=col_rate, aggfunc='mean')
fig, ax = plt.subplots(figsize=(20,12))
sns.heatmap(state_monthly, cmap='RdYlGn_r', ax=ax, linewidths=0.3, linecolor='#e0e0e0',
            cbar_kws={'label':'Unemployment Rate (%)'})
cols = list(state_monthly.columns)
covid_col = next((i for i,c in enumerate(cols) if str(c)=='2020-04'), None)
if covid_col: ax.axvline(covid_col, color=COVID, lw=2.5, linestyle='--')
ax.set_title('State-wise Unemployment Rate Heatmap', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=7)
plt.tight_layout(); plt.show()

## 6. 🏙️ Urban vs Rural Analysis

In [ ]:
if 'Area' in df.columns:
    area_time = df.groupby(['Date','Area'])[col_rate].mean().reset_index()
    fig, axes = plt.subplots(1,2,figsize=(15,5))
    for area,color in [('Urban','#E91E63'),('Rural','#009688')]:
        sub = area_time[area_time['Area']==area]
        axes[0].plot(sub['Date'],sub[col_rate],label=area,color=color,lw=2,marker='o',markersize=4)
    axes[0].axvspan(pd.Timestamp('2020-04-01'),pd.Timestamp('2020-05-31'),color=COVID,alpha=0.12)
    axes[0].set_title('Unemployment Over Time by Area'); axes[0].set_ylabel('%'); axes[0].legend()
    area_period = df.groupby(['Covid_Period','Area'])[col_rate].mean().reset_index()
    area_pivot = area_period.pivot(index='Covid_Period',columns='Area',values=col_rate)
    area_pivot.loc[['Pre-Covid','Lockdown','Recovery']].plot(kind='bar',ax=axes[1],
        color=['#E91E63','#009688'],edgecolor='white',width=0.6)
    axes[1].set_title('Mean Rate by Period & Area'); axes[1].set_ylabel('%')
    axes[1].tick_params(axis='x',rotation=0); axes[1].grid(axis='y',alpha=0.3)
    plt.tight_layout(); plt.show()

## 7. 📉 Recovery Trajectory & Trend Analysis

In [ ]:
nat = df.groupby('Date')[col_rate].mean().reset_index()
pre_mean = nat[nat['Date']<'2020-04-01'][col_rate].mean()
peak_val = nat[col_rate].max()

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(nat['Date'],nat[col_rate],color='#424242',lw=2.5,zorder=5,label='National Avg')
ax.fill_between(nat['Date'],nat[col_rate],
    where=(nat['Date']>='2020-04-01')&(nat['Date']<='2020-05-31'),color=COVID,alpha=0.5,label='Lockdown Spike')
ax.fill_between(nat['Date'],nat[col_rate],where=nat['Date']>'2020-05-31',color=POST,alpha=0.3,label='Recovery')
ax.axhline(pre_mean,color=PRE,lw=1.5,linestyle='--',label=f'Pre-Covid Avg: {pre_mean:.1f}%')
ax.axhline(peak_val,color=COVID,lw=1.5,linestyle=':',label=f'Peak: {peak_val:.1f}%')
ax.set_title('Unemployment Recovery Trajectory Post-Lockdown', fontsize=13, fontweight='bold')
ax.set_ylabel('%'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. 💡 Key Insights & Policy Recommendations

In [ ]:
pre  = df[df['Covid_Period']=='Pre-Covid'].groupby('Region')[col_rate].mean()
lock = df[df['Covid_Period']=='Lockdown'].groupby('Region')[col_rate].mean()
impact = (lock - pre).dropna().sort_values(ascending=False)

print('='*55)
print('KEY INSIGHTS — UNEMPLOYMENT ANALYSIS IN INDIA')
print('='*55)
print(f'\n📊 NATIONAL STATISTICS:')
print(f'  Pre-Covid Average Rate : {df[df["Covid_Period"]=="Pre-Covid"][col_rate].mean():.2f}%')
print(f'  Lockdown Average Rate  : {df[df["Covid_Period"]=="Lockdown"][col_rate].mean():.2f}%')
print(f'  Recovery Average Rate  : {df[df["Covid_Period"]=="Recovery"][col_rate].mean():.2f}%')
print(f'  Peak Rate              : {df[col_rate].max():.2f}%')
print(f'\n🔴 COVID-19 IMPACT:')
print(f'  Rate increase during lockdown: +{df[df["Covid_Period"]=="Lockdown"][col_rate].mean() - df[df["Covid_Period"]=="Pre-Covid"][col_rate].mean():.1f} percentage points')
print(f'  Most impacted state: {impact.idxmax()} (+{impact.max():.1f}pp)')
print(f'\n🟢 RECOVERY:')
rec = df[df['Covid_Period']=='Recovery'].groupby('Region')[col_rate].mean()
rec_from_lock = (lock - rec).dropna()
print(f'  Fastest recovering state: {rec_from_lock.idxmax()} (-{rec_from_lock.max():.1f}pp)')
print(f'\n💡 POLICY INSIGHTS:')
insights = [
    'Urban areas suffered sharper spikes than rural during lockdown',
    'States with high pre-covid rates (>15%) need targeted job schemes',
    'Seasonal Q1 dip and Q4 recovery suggests agricultural employment impact',
    'Labour Participation Rate declined alongside unemployment — discouraged workers',
    'Recovery was uneven across states — federal coordination needed',
]
for ins in insights: print(f'  • {ins}')